# Baselines, models and honest chronology

A target row dated **d** can use weather through **d−1** and calendar facts for **d**.
Climatology, scalers and imputers are fitted on the current fold's training rows.
Validation chooses shared hyperparameters using the equal-weight mean site MAE.
Test models refit on train ∪ validation, using the selected configuration.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image
get_ipython().run_line_magic('matplotlib', 'inline')
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT))
from skillmap.config import PROCESSED, FIGURES, SPLITS, WEATHER_COLUMNS
findings = json.loads((PROCESSED / 'findings.json').read_text())
sites = pd.read_csv(PROCESSED / 'sites.csv', keep_default_na=False)
predictions = pd.read_parquet(PROCESSED / 'predictions.parquet')
selected = sites[sites.site_id.isin(findings['site_ids'])]
display(Markdown(f"This run evaluates **{findings['n_sites']} complete sites**. "
                 f"The full sampling frame contains **{len(sites)} sites**."))

This run evaluates **200 complete sites**. The full sampling frame contains **200 sites**.

In [2]:
display(pd.DataFrame(SPLITS, index=['start', 'end']).T)
from skillmap.features import build_features, FULL
from skillmap.evaluate import _prepare_site
daily = pd.read_parquet(PROCESSED / 'daily.parquet')
sample = daily[daily.site_id == findings['site_ids'][0]].copy()
table = build_features(sample)
fold = _prepare_site(table)
demo = fold.val.head(5).copy()
demo['latest_weather_date'] = demo.date - pd.Timedelta(days=1)
display(demo[['date','latest_weather_date','y_true','tmax_lag1',
              'tmax_lag7','tmax_anomaly_lag1','climatology_target']])
target = demo.date.iloc[0]
changed = sample.copy()
changed.loc[changed.date >= target, WEATHER_COLUMNS] += 1000
changed_features = build_features(changed)
protected = table.date <= target
pd.testing.assert_frame_equal(table.loc[protected, FULL],
                              changed_features.loc[protected, FULL])
print('Changing weather on d and all later dates leaves predictors through d unchanged.')

,start,end
train,2015-01-01,2021-12-31
val,2022-01-01,2023-12-31
test,2024-01-01,2025-12-31


,date,latest_weather_date,y_true,tmax_lag1,tmax_lag7,tmax_anomaly_lag1,climatology_target
2557,2022-01-01,2021-12-31,25.7,21.9,26.8,-4.234857,26.203902
2558,2022-01-02,2022-01-01,23.9,25.7,29.6,-0.503902,26.258317
2559,2022-01-03,2022-01-02,25.9,23.9,30.1,-2.358317,26.311929
2560,2022-01-04,2022-01-03,28.2,25.9,25.2,-0.411929,26.364673
2561,2022-01-05,2022-01-04,24.1,28.2,23.7,1.835327,26.416482


Changing weather on d and all later dates leaves predictors through d unchanged.


## Models and selected configurations

Persistence predicts yesterday's Tmax. Climatology fits three annual harmonics per site.
Local ridge uses median imputation and scaling; local HGB handles missing predictors directly.
Pooled HGB learns temperature anomalies across sites using the same lag predictors, latitude,
absolute latitude, elevation, categorical Köppen group and fold-fitted climatology for d.
Climatology is added back after predicting an anomaly.

The best baseline is selected separately per site on paired validation dates, and that
choice is retained on test. Validation numbers below are selection diagnostics.

In [3]:
experiment = findings['experiment']
display(pd.DataFrame(experiment['selected_params']).T)
scores = []
for feature_group, models in experiment['validation_scores'].items():
    for model, candidates in models.items():
        for candidate in candidates:
            scores.append(dict(feature_group=feature_group, model=model, **candidate))
display(pd.DataFrame(scores))
display(pd.DataFrame(findings['grid_edges']))
metrics = pd.read_csv(PROCESSED / 'metrics_site.csv')
display(metrics.groupby(['model','split']).mae.mean().unstack('split'))
display(pd.DataFrame(experiment['fit_windows']))
minutes = experiment['runtime_seconds'] / 60
display(Markdown(f"Measured experiment runtime: **{minutes:.2f} min**, "
                 f"with **{experiment['n_jobs']} jobs**."))

,linear,hgb,hgb_pooled
temp_calendar,{'alpha': 0.001},"{'max_iter': 200, 'max_leaf_nodes': 7}","{'max_iter': 400, 'max_leaf_nodes': 15}"
full,{'alpha': 10.0},"{'max_iter': 400, 'max_leaf_nodes': 7}","{'max_iter': 400, 'max_leaf_nodes': 15}"


,feature_group,model,params,mae
0,temp_calendar,linear,{'alpha': 0.001},1.725654
1,temp_calendar,linear,{'alpha': 0.01},1.725913
2,temp_calendar,linear,{'alpha': 0.1},1.726031
3,temp_calendar,linear,{'alpha': 1.0},1.726044
4,temp_calendar,linear,{'alpha': 10.0},1.727649
5,temp_calendar,hgb,"{'max_iter': 200, 'max_leaf_nodes': 7}",1.730608
6,temp_calendar,hgb,"{'max_iter': 200, 'max_leaf_nodes': 15}",1.737098
7,temp_calendar,hgb,"{'max_iter': 400, 'max_leaf_nodes': 7}",1.733737
8,temp_calendar,hgb,"{'max_iter': 400, 'max_leaf_nodes': 15}",1.743668
9,temp_calendar,hgb,"{'max_iter': 700, 'max_leaf_nodes': 7}",1.738605


,feature_group,model,parameter,value,edge
0,temp_calendar,linear,alpha,0.001,low
1,temp_calendar,hgb,max_iter,200.000,low
2,temp_calendar,hgb,max_leaf_nodes,7.000,low
3,temp_calendar,hgb_pooled,max_iter,400.000,high
4,full,linear,alpha,10.000,high
5,full,hgb,max_leaf_nodes,7.000,low
6,full,hgb_pooled,max_iter,400.000,high


split,test,val
model,,
climatology,2.554329,2.581191
hgb,1.641165,1.649778
hgb_pooled,1.653328,1.650097
linear,1.667256,1.681396
persistence,1.860378,1.857666


,val,test
0,2015-01-01,2015-01-01
1,2021-12-31,2023-12-31


Measured experiment runtime: **27.18 min**, with **8 jobs**.

## Uncertainty

Skill is `1 − model_MAE / reference_MAE`; positive values favor ML. Paired moving-block
bootstrap resamples contiguous calendar-date blocks. Missing dates stay in the index,
and each comparison uses common finite support. Group estimates weight sites equally
and resample the same date blocks across sites to preserve contemporaneous dependence.
Percentile intervals condition on the fitted, selected models. They do not include model
selection uncertainty. Exploratory driver intervals instead bootstrap entire sites.

In [4]:
display(pd.Series({k: experiment[k] for k in ['block_days','bootstrap_reps','seed']}))
skill = pd.read_csv(PROCESSED / 'skill_group.csv')
display(skill.query("koppen_group == 'ALL' and reference == 'best_baseline'"))

block_days          14
bootstrap_reps    1000
seed                42
dtype: int64

,koppen_group,split,model,reference,reference_model,mae_model,mae_ref,delta_mae,delta_ci_low,delta_ci_high,skill,skill_ci_low,skill_ci_high,n_sites
17,ALL,val,linear,best_baseline,per_site,1.681396,1.849809,0.168413,0.154343,0.181737,0.091044,0.084679,0.097346,200
35,ALL,val,hgb,best_baseline,per_site,1.649778,1.849809,0.200031,0.188096,0.211116,0.108136,0.102713,0.113137,200
53,ALL,val,hgb_pooled,best_baseline,per_site,1.650097,1.849809,0.199712,0.190371,0.208376,0.107963,0.103499,0.112590,200
71,ALL,test,linear,best_baseline,per_site,1.667256,1.855198,0.187942,0.173034,0.199812,0.101305,0.094716,0.106662,200
89,ALL,test,hgb,best_baseline,per_site,1.641165,1.855198,0.214033,0.202275,0.224801,0.115369,0.110479,0.119831,200
107,ALL,test,hgb_pooled,best_baseline,per_site,1.653328,1.855198,0.201870,0.191753,0.211834,0.108813,0.104255,0.113305,200
